# einops-einsum — ex8: batched bilinear form y = x^T A x over a 2D grid, with heatmap viz

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. Running the final beacon cell reports progress against the `Einops: Deep Learning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-einsum`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.einsum — quick refresher

`einsum(*tensors, pattern)` performs sum-contraction over named indices:
1. **Elementwise** — `'i j, i j -> i j'` multiplies pointwise (no reduction).
2. **Matmul** — `'i k, k j -> i j'` contracts the shared `k` (sum-reduce).
3. **Batched** — `'b i k, b k j -> b i j'` carries `b` through, contracts `k`.
4. **Three operands** — `'i j, j k, k l -> i l'` chains two contractions; the optimizer picks pairing order.

**The two rules:**
- An index that appears on input AND output → preserved (broadcast-like).
- An index that appears on input but NOT on output → sum-contracted.

### Exercise 8 — batched bilinear form y = x^T A x over a 2D grid, with heatmap viz

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Use a single einsum pattern with three operands (x, A, x) to compute the quadratic form y = x^T A x for a batch of vectors, then visualize it as a 2D heatmap over a grid of x values.
> Keywords: bilinear, quadratic-form, broadcast, visualization
> ```

**KCs targeted:** `einsum-three-operand-contraction`, `einsum-batched`, `einsum-shared-tensor-broadcast`

Implement `ex8_batched_bilinear(x, A)`.

Inputs:
- `x`: `(N, D)` — batch of N row-vectors.
- `A`: `(D, D)` — a **shared** matrix used for every x in the batch.

Output: `(N,)` — the vector of bilinear forms `y[n] = x[n] @ A @ x[n]` (a scalar per row).

Use a **single** `einops.einsum` call passing three operands `(x, A, x)` and one pattern that contracts both indices of `A`. Do not reshape, transpose, or call `@` / `matmul`.

The test cell additionally evaluates your function on a `49 x 49` grid of `x = (x0, x1)` in `[-2, 2]^2` with a fixed `A`, and plots the resulting scalar field as a heatmap. For a positive-definite `A` you should see concentric elliptical contours around the origin.

In [ ]:
def ex8_batched_bilinear(x: Tensor, A: Tensor) -> Tensor:
    # x appears twice with different contraction indices (i, j); A contracts both.
    # n is preserved (batch); i and j are both contracted (no shared output).
    return einsum(x, A, x, 'n i, i j, n j -> n')


<details><summary>Solution</summary>

```python
def ex8_batched_bilinear(x: Tensor, A: Tensor) -> Tensor:
    # x appears twice with different contraction indices (i, j); A contracts both.
    # n is preserved (batch); i and j are both contracted (no shared output).
    return einsum(x, A, x, 'n i, i j, n j -> n')
```

**The trick: the same tensor twice with different indices.** einsum lets you pass the same tensor more than once — the index name on each position is what disambiguates. Here `x` appears with index `i` first and `j` second; the contraction `i j` is exactly the inner double-sum that defines `x^T A x`.

**Reading the pattern.**
- `n` appears in both `x` operands and on the output → preserved (batch axis).
- `i` appears in the first `x` and in `A`, not on the output → contracted.
- `j` appears in the second `x` and in `A`, not on the output → contracted.
- `A` has no `n` → it's broadcast across the batch (shared matrix).

**Reading the heatmap.** With `A = [[1, 0.6], [0.6, 2]]` (positive definite, eigenvalues both positive), the level sets are ellipses centered at the origin. The tilt comes from the off-diagonal `0.6`; the vertical squish comes from the larger eigenvalue along the y-axis (`A[1,1] = 2 > A[0,0] = 1`).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()